In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from langchain_neo4j import Neo4jGraph
from neo4j import GraphDatabase, Session
import pandas as pd
import numpy as np  # For NaN handling

In [2]:
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

In [3]:
class CustomNeo4jGraph(Neo4jGraph):
    def query(self, query, params=None, **kwargs):
        """
        Override to use session.run directly, bypassing execute_query.
        """
        with self._driver.session(**kwargs) as session:
            result = session.run(query, params or {})
            return [dict(record) for record in result]  # Convert to list of dicts like original

# Instantiate with refresh disabled
graph = CustomNeo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database="neo4j",
    refresh_schema=False  # Avoid initial errors
)

# Strong clear using direct driver to ensure all nodes/relationships are removed (handles residuals)
direct_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [4]:
def clear_all(tx):
    tx.run("MATCH (n) DETACH DELETE n")

with direct_driver.session() as session:
    session.execute_write(clear_all)
direct_driver.close()
print("Graph cleared completely with direct driver.")

Graph cleared completely with direct driver.


In [5]:
graph.refresh_schema()

In [6]:
df = pd.read_csv('movies_small.csv')

In [7]:
# Convert and validate numeric columns
df['movieId'] = pd.to_numeric(df['movieId'], errors='coerce').fillna(0).astype(int)
df = df[df['movieId'] > 0]  # Drop invalid IDs (e.g., 0 or NaN)
df['title'] = df['title'].astype(str).str.strip()
df = df[df['title'] != 'nan']  # Treat NaN strings as invalid
df = df[df['title'] != '']     # Drop empty titles
df['released'] = pd.to_numeric(df['released'], errors='coerce').astype('Int64')  # Nullable int for NaNs
df = df.dropna(subset=['released'])  # Drop null released
df['imdbRating'] = pd.to_numeric(df['imdbRating'], errors='coerce')
df = df.dropna(subset=['imdbRating'])  # Drop null ratings

# Clean delimited fields (director, actors, genres): Treat NaN as empty, drop fully empty rows if needed
for col in ['director', 'actors', 'genres']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = np.where(df[col] == 'nan', '', df[col])  # NaN -> empty string
        df = df[df[col] != '']  # Optional: Drop rows with fully empty delimited fields

print(f"Cleaned DataFrame: {len(df)} rows, ready for import")
print(df.head(3).to_string(index=False))  # Sample output for verification

Cleaned DataFrame: 20 rows, ready for import
 movieId           title  released                                             actors          director                 genres  imdbRating
       1       Inception      2010  Leonardo DiCaprio|Joseph Gordon-Levitt|Ellen Page Christopher Nolan Sci-Fi|Action|Thriller         8.8
       2 The Dark Knight      2008          Christian Bale|Heath Ledger|Aaron Eckhart Christopher Nolan     Action|Crime|Drama         9.0
       3    Interstellar      2014 Matthew McConaughey|Anne Hathaway|Jessica Chastain Christopher Nolan Sci-Fi|Adventure|Drama         8.6


In [8]:
import pandas as pd  # For handling NaN and string parsing

def import_row(row_data):
    """
    Imports one movie record (row) into Neo4j with nodes and relationships:
    - Movie node (MERGE by movieId)
    - Person nodes (directors, actors)
    - Genre nodes
    - Relationships: DIRECTED, ACTED_IN, IN_GENRE
    """

    # --- Extract and validate input values ---
    try:
        movie_id = int(row_data['movieId'])              # Ensure movieId is integer
        title = str(row_data['title']).strip()           # Clean up title string
        released = int(row_data['released'])             # Year of release
        imdb_rating = float(row_data['imdbRating'])      # Ensure rating is float
    except (ValueError, TypeError) as e:
        print(f"Type error in row movieId={row_data.get('movieId', 'N/A')}: {e}, skipping")
        return  # Skip invalid rows

    # Skip if any field is missing or invalid
    if not all([title and title != 'nan', movie_id > 0, released > 0, imdb_rating > 0]):
        print(f"Invalid params for movieId={movie_id} (title='{title}'), skipping")
        return

    # Parameters dictionary for Cypher queries
    params = {
        'movieId': movie_id,
        'title': title,
        'released': released,
        'imdbRating': imdb_rating
    }

    # --- Create or update Movie node ---
    movie_cypher = """
    MERGE (m:Movie {movieId: $movieId})
    SET m.title = $title,
        m.released = $released,
        m.imdbRating = $imdbRating,
        m.name = $title
    RETURN m.movieId AS id, m.title AS title, m.name AS name, m.movieId AS prop_id
    """
    result = graph.query(movie_cypher, params)
    if not result or not result[0].get('id'):
        print(f"Failed to create/update movieId: {movie_id}")
        return
    print(f"Movie created/updated: {result[0]['title']} (ID: {result[0]['id']}, Name: {result[0]['name']})")

    # --- Verify Movie node creation ---
    verify_movie = graph.query("""
    MATCH (m:Movie {movieId: $movieId})
    RETURN m.name AS name, m.movieId AS prop_id, m.title AS title
    """, {'movieId': movie_id})
    if not verify_movie:
        print(f"ERROR: Movie {movie_id} not found after SET—abort rels")
        return
    print(f"  Verified Movie {movie_id} exists with name: {verify_movie[0]['name']}, prop_id: {verify_movie[0]['prop_id']}")

    # --- Create DIRECTED relationships ---
    director_str = str(row_data.get('director', '')).strip()
    added_directors = 0
    if director_str and director_str.lower() != 'nan':
        for director in director_str.split('|'):
            director_name = director.strip()
            if director_name:
                director_cypher = """
                MERGE (d:Person {name: $name})
                WITH d
                MATCH (m:Movie {movieId: $movieId})
                MERGE (d)-[:DIRECTED]->(m)
                RETURN 1 AS success
                """
                rel_result = graph.query(director_cypher, {'name': director_name, 'movieId': movie_id})
                if rel_result and rel_result[0].get('success') == 1:
                    print(f"  Added director: '{director_name}' for '{title}'")
                    added_directors += 1
    else:
        print(f"  No valid directors for '{title}'")
    print(f"  Total directors added: {added_directors}")

    # --- Create ACTED_IN relationships ---
    actors_str = str(row_data.get('actors', '')).strip()
    added_actors = 0
    if actors_str and actors_str.lower() != 'nan':
        for actor in actors_str.split('|'):
            actor_name = actor.strip()
            if actor_name:
                actor_cypher = """
                MERGE (a:Person {name: $name})
                WITH a
                MATCH (m:Movie {movieId: $movieId})
                MERGE (a)-[:ACTED_IN]->(m)
                RETURN 1 AS success
                """
                rel_result = graph.query(actor_cypher, {'name': actor_name, 'movieId': movie_id})
                if rel_result and rel_result[0].get('success') == 1:
                    print(f"  Added actor: '{actor_name}' for '{title}'")
                    added_actors += 1
    else:
        print(f"  No valid actors for '{title}'")
    print(f"  Total actors added: {added_actors}")

    # --- Create IN_GENRE relationships ---
    genres_str = str(row_data.get('genres', '')).strip()
    added_genres = 0
    if genres_str and genres_str.lower() != 'nan':
        for genre in genres_str.split('|'):
            genre_name = genre.strip()
            if genre_name:
                genre_cypher = """
                MERGE (g:Genre {name: $name})
                WITH g
                MATCH (m:Movie {movieId: $movieId})
                MERGE (m)-[:IN_GENRE]->(g)
                RETURN 1 AS success
                """
                rel_result = graph.query(genre_cypher, {'name': genre_name, 'movieId': movie_id})
                if rel_result and rel_result[0].get('success') == 1:
                    print(f"  Added genre: '{genre_name}' for '{title}'")
                    added_genres += 1
    else:
        print(f"  No valid genres for '{title}'")
    print(f"  Total genres added: {added_genres}")


In [9]:
batch_size = 5
for i in range(0, len(df), batch_size):
    batch_df = df.iloc[i:i + batch_size]
    print(f"\n--- Starting Batch {i // batch_size + 1} ({len(batch_df)} rows) ---")
    for _, row in batch_df.iterrows():
        import_row(row)
    print(f"Batch {i // batch_size + 1} completed\n")

print("Full import completed—check logs for rel confirmations!")


--- Starting Batch 1 (5 rows) ---
Movie created/updated: Inception (ID: 1, Name: Inception)
  Verified Movie 1 exists with name: Inception, prop_id: 1
  Added director: 'Christopher Nolan' for 'Inception'
  Total directors added: 1
  Added actor: 'Leonardo DiCaprio' for 'Inception'
  Added actor: 'Joseph Gordon-Levitt' for 'Inception'
  Added actor: 'Ellen Page' for 'Inception'
  Total actors added: 3
  Added genre: 'Sci-Fi' for 'Inception'
  Added genre: 'Action' for 'Inception'
  Added genre: 'Thriller' for 'Inception'
  Total genres added: 3
Movie created/updated: The Dark Knight (ID: 2, Name: The Dark Knight)
  Verified Movie 2 exists with name: The Dark Knight, prop_id: 2
  Added director: 'Christopher Nolan' for 'The Dark Knight'
  Total directors added: 1
  Added actor: 'Christian Bale' for 'The Dark Knight'
  Added actor: 'Heath Ledger' for 'The Dark Knight'
  Added actor: 'Aaron Eckhart' for 'The Dark Knight'
  Total actors added: 3
  Added genre: 'Action' for 'The Dark Knigh

In [10]:
from langchain_groq import ChatGroq
from langchain_neo4j import GraphCypherQAChain
from langchain.prompts import PromptTemplate

In [11]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env. Add it from groq.com.")

In [12]:
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="openai/gpt-oss-120b",  # Or "mixtral-8x7b-32768" for better reasoning
    temperature=0  # Low for deterministic Cypher generation
)

In [13]:
CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],  # Only these—chain provides schema/question from query
    template="""
Based on the Neo4j schema {schema}, write a Cypher query to answer the question: {question}.
Follow these schema guidelines strictly to generate accurate Cypher:
- Movie nodes: Label Movie. Properties: movieId (int, unique), title (string), name (string alias=title), released (int), imdbRating (float).
  - Match Movies with: (m:Movie {{movieId: num}}) or (m:Movie {{title: 'Exact Title'}}) or (m:Movie {{name: 'Exact Title'}}).
- Person nodes: Label Person. Property: name (string, unique).
  - Relationships: (person)-[:DIRECTED]->(m:Movie) for directors; (person)-[:ACTED_IN]->(m:Movie) for actors.
- Genre nodes: Label Genre. Property: name (string, unique).
  - Relationships: (m:Movie)-[:IN_GENRE]->(g:Genre).
- Best practices:
  - For directors/actors: MATCH (p:Person {{name: 'Exact Name'}})-[rel]->(m:Movie) WHERE rel = :DIRECTED or :ACTED_IN.
  - Filters: Use WHERE m.released > 2000, m.imdbRating DESC.
  - Collections: Use collect(DISTINCT prop) AS list to avoid duplicates (e.g., actors).
  - Top N: WITH m ORDER BY m.imdbRating DESC LIMIT 3; then MATCH further.
  - Return: Relevant props like p.name, m.title, m.imdbRating, collect(...) AS summary.
- Ensure exact prop names/case; use DISTINCT where needed.
Output ONLY the Cypher query—no explanations or code blocks.
"""
)

In [14]:
try:
    graph.refresh_schema()
    print("Schema refreshed successfully.")
except Exception as e:
    print(f"Schema refresh skipped (expected with custom): {e}")

# Create the chain: Uses langchain_neo4j version, compatible with CustomNeo4jGraph
graph_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,  # Works with your custom graph
    cypher_prompt= CYPHER_GENERATION_PROMPT,
    verbose=True,  # Debug: Shows Cypher, results, NL generation
    top_k=10,  # Top results limit
    allow_dangerous_requests=True  # Required for Neo4j chains; enables full query perms (use cautiously)
)

print("RAG Chain created successfully with langchain_neo4j!")

Schema refreshed successfully.
RAG Chain created successfully with langchain_neo4j!


In [15]:
# Original Tests (1-4: Basic to Complex, confirming graph/chain setup)

# Test 1: Director query
response = graph_chain.invoke({"query": "Who directed The Dark Knight?"})
print("Response:", response['result'])  # E.g., "Christopher Nolan directed The Dark Knight."

# Test 2: Actors and rating
response = graph_chain.invoke({"query": "Actors in Inception with imdbRating?"})
print("Response:", response['result'])  # E.g., "Actors: Leonardo DiCaprio, etc. Rating: 8.8"

# Test 3: Genres
response = graph_chain.invoke({"query": "Genres of movies directed by Christopher Nolan after 2000"})
print("Response:", response['result'])  # E.g., "Sci-Fi, Action, Thriller for Inception; etc."

# Test 4: Complex
response = graph_chain.invoke({"query": "Top 3 highest-rated Nolan movies and their actors"})
print("Response:", response['result'])  # LLM summarizes with rankings

# Additional Tests (5-14: Simple Lookups, Filters, Aggregates, Multi-Joins—validate full coverage)

# Test 5: Simple Movie Props (release year and rating)
response = graph_chain.invoke({"query": "What is the release year and IMDb rating of Inception?"})
print("Response:", response['result'])  # E.g., "Inception released in 2010 with rating 8.8"

# Test 6: Genre List for Specific Movie
response = graph_chain.invoke({"query": "List all genres for The Dark Knight."})
print("Response:", response['result'])  # E.g., "Genres: Action, Crime, Drama, Thriller"

# Test 7: Main Actors for a Movie
response = graph_chain.invoke({"query": "Who are the main actors in Interstellar?"})
print("Response:", response['result'])  # E.g., "Actors: Matthew McConaughey, Anne Hathaway, Jessica Chastain, etc."

# Test 8: Filtered Movies (year and rating)
response = graph_chain.invoke({"query": "Which movies released after 2010 have an IMDb rating above 8.0?"})
print("Response:", response['result'])  # E.g., "Movies: Inception (8.8, 2010)—adjust year if needed; higher-rated like Interstellar"

# Test 9: Director Stats (count and average rating)
response = graph_chain.invoke({"query": "How many movies did Christopher Nolan direct, and what are their average ratings?"})
print("Response:", response['result'])  # E.g., "Nolan directed 10 movies with average rating 8.5"

# Test 10: Frequent Actors (HAVING count > 3)
response = graph_chain.invoke({"query": "Find actors who appeared in more than 3 Nolan movies."})
print("Response:", response['result'])  # E.g., "Actors: Michael Caine (5 movies), etc."

# Test 11: Top Movies by Genre
response = graph_chain.invoke({"query": "List the top 5 highest-rated movies in the Sci-Fi genre and their directors."})
print("Response:", response['result'])  # E.g., "1. Inception (8.8, Christopher Nolan); 2. Interstellar (8.6, Christopher Nolan)"

# Test 12: Frequent Genres for Director
response = graph_chain.invoke({"query": "Find genres that appear in at least 5 movies directed by Christopher Nolan."})
print("Response:", response['result'])  # E.g., "Common genres: Drama (7 movies), Action (5 movies)"

# Test 13: Highest-Rated per Genre (GROUP BY with max)
response = graph_chain.invoke({"query": "What is the highest-rated movie per genre, including its director and rating?"})
print("Response:", response['result'])  # E.g., "Action: The Dark Knight (9.0, Christopher Nolan); Sci-Fi: Inception (8.8, Christopher Nolan)"




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: 'The Dark Knight'})
RETURN p.name AS director, m.title AS movie;
Full Context:
[{'director': 'Christopher Nolan', 'movie': 'The Dark Knight'}]

> Finished chain.
Response: Christopher Nolan directed The Dark Knight.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: 'Inception'})
MATCH (p:Person)-[:ACTED_IN]->(m)
RETURN p.name AS actor, m.title AS movie, m.imdbRating AS imdbRating
ORDER BY p.name
Full Context:
[{'actor': 'Ellen Page', 'movie': 'Inception', 'imdbRating': 8.8}, {'actor': 'Joseph Gordon-Levitt', 'movie': 'Inception', 'imdbRating': 8.8}, {'actor': 'Leonardo DiCaprio', 'movie': 'Inception', 'imdbRating': 8.8}]

> Finished chain.
Response: Ellen Page, Joseph Gordon‑Levitt, Leonardo DiCaprio have an IMDb rating of 8.8.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {name: 'Christopher Nolan'})-[:DIRECT

## Prompt Strategies : FEW-SHOT Prompting

In [16]:
print(graph_chain.graph_schema)

Node properties:
Person {name: STRING}
Movie {name: STRING, title: STRING, released: INTEGER, movieId: INTEGER, imdbRating: FLOAT}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


In [20]:
complex_tests_examples = [
    {
        "question": "Recommend co-actors for Leonardo DiCaprio that he hasn't worked with, but who his co-actors have collaborated with, ordered by collaboration strength.",
        "query": """
        MATCH (diCaprio:Person {{name: 'Leonardo DiCaprio'}})-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(coActors:Person),
        (coActors)-[:ACTED_IN]->(m2:Movie)<-[:ACTED_IN]-(recommended:Person)
        WHERE NOT (diCaprio)-[:ACTED_IN]->()<-[:ACTED_IN]-(recommended) AND diCaprio <> recommended
        RETURN recommended.name AS recommended, count(recommended) AS strength
        ORDER BY strength DESC
        LIMIT 5
        """
        # Expected: Rows like Tom Hanks (strength: 2+ via shared co-actors), context for LLM summary: "Recommendations: Tom Hanks (strength 3), etc."
    },
    {
        "question": "Find co-actors who could introduce Christopher Nolan to Leonardo DiCaprio through shared movies (but they haven't collaborated directly).",
        "query": """
        MATCH (nolan:Person {{name: 'Christopher Nolan'}})-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(coActor:Person),
        (coActor)-[:ACTED_IN]->(m2:Movie)<-[:DIRECTED OR ACTED_IN]-(diCaprio:Person {{name: 'Leonardo DiCaprio'}})
        WHERE NOT (nolan)-[:DIRECTED]->()<-[:ACTED_IN]-(diCaprio)
        RETURN DISTINCT coActor.name AS coActor
        ORDER BY coActor.name
        """
        # Expected: Rows like Joseph Gordon-Levitt (Inception dir Nolan, acted with DiCaprio elsewhere), response: "Co-actors: Joseph Gordon-Levitt, etc."
    },
    {
        "question": "Which actors have the longest streak of consecutive release years in movies (at least 3 years), and list their movies in that streak?",
        "query": """
        MATCH (actor:Person)-[:ACTED_IN]->(m:Movie)
        WITH actor, collect(DISTINCT m.released) AS years
        UNWIND apoc.coll.sort(years) AS sorted_years  // Requires APOC; sort years
        WITH actor, sorted_years,
             [i IN range(0, size(sorted_years)-2) WHERE sorted_years[i+1] = sorted_years[i]+1 | sorted_years[i]..sorted_years[i+1]] AS streaks
        WITH actor, max(size(streak)) AS max_streak_length, [streak IN streaks WHERE size(streak) = max_streak_length | streak] AS best_streaks
        WHERE max_streak_length >= 3
        RETURN actor.name, max_streak_length, best_streaks[0] AS streak_years,  // First best streak
               [(y) IN best_streaks[0] | [m IN [(actor)-[:ACTED_IN]->(mm:Movie) WHERE mm.released = y | mm.title]] | m] AS movies_in_streak
        ORDER BY max_streak_length DESC
        LIMIT 3
        """
        # Expected: Actors like Michael Caine (Nolan streaks: 2005-2012, 8 years; movies: Batman series), note: APOC plugin for coll.sort/range—adapt if no APOC.
    },
    {
        "question": "Identify genres that are most common in high-rated movies (above 8.5) directed by Christopher Nolan, with count and average rating per genre.",
        "query": """
        MATCH (nolan:Person {{name: 'Christopher Nolan'}})-[:DIRECTED]->(m:Movie)-[:IN_GENRE]->(g:Genre)
        WHERE m.imdbRating > 8.5
        WITH g.name AS genre, count(m) AS movie_count, avg(m.imdbRating) AS avg_rating
        RETURN genre, movie_count, avg_rating
        ORDER BY movie_count DESC, avg_rating DESC
        """
        # Expected: Rows like 'Sci-Fi' (2 movies, avg 8.7: Inception/Interstellar), 'Action' (2, avg 9.0: Dark Knight), response: "Common: Sci-Fi (2, 8.7); Action (2, 9.0)."
    },
    {
        "question": "Find pairs of movies that share at least 3 actors and have similar ratings (diff < 1.0), including the shared actors.",
        "query": """
        MATCH (m1:Movie)-[:ACTED_IN]-(a:Person)-[:ACTED_IN]-(m2:Movie)
        WHERE m1.movieId < m2.movieId AND abs(m1.imdbRating - m2.imdbRating) < 1.0
        WITH m1, m2, collect(DISTINCT a.name) AS shared_actors
        WHERE size(shared_actors) >= 3
        RETURN m1.title AS movie1, m2.title AS movie2, m1.imdbRating AS rating1, m2.imdbRating AS rating2,
               shared_actors, size(shared_actors) AS shared_count
        ORDER BY shared_count DESC
        LIMIT 5
        """
        # Expected: Pairs like Inception/Dark Knight (shared: e.g., Cillian Murphy if data; adjust for actual), response: "Pairs: Inception & Interstellar (4 shared, ratings 8.8/8.6: actors X,Y,Z)."
    },
    {
        "question": "What is the shortest path (fewest hops) between Tom Hanks and Leonardo DiCaprio via co-actors or directors, and list the path?",
        "query": """
        MATCH path = shortestPath( (hanks:Person {{name: 'Tom Hanks'}})-[*]-(diCaprio:Person {{name: 'Leonardo DiCaprio'}}) )
        WHERE ALL(r IN relationships(path) WHERE type(r) IN ['ACTED_IN', 'DIRECTED'])
        RETURN path, length(path) AS hops
        ORDER BY hops ASC
        LIMIT 1
        """
        # Expected: Short path like Hanks -ACTED_IN-> Forrest Gump -DIRECTED-> (someone) -ACTED_IN-> DiCaprio movie (hops: 3-5), response: "Shortest path: 3 hops via co-actor X."
    },
    {
        "question": "Rank directors by the diversity of genres in their filmography (unique genres count > 5), with total movies and unique genres list.",
        "query": """
        MATCH (d:Person)-[:DIRECTED]->(m:Movie)-[:IN_GENRE]->(g:Genre)
        WITH d, collect(DISTINCT g.name) AS genres, count(DISTINCT m) AS movie_count
        WHERE size(genres) > 5
        RETURN d.name AS director, movie_count, size(genres) AS genre_diversity, genres
        ORDER BY genre_diversity DESC, movie_count DESC
        LIMIT 3
        """
        # Expected: Directors like Steven Spielberg (diversity 10+, genres: Action, Drama, Sci-Fi,...), but for Nolan data: Nolan (6-8: Sci-Fi, Action, Thriller, Drama), response: "Top: Christopher Nolan (8 movies, 6 genres: Sci-Fi, Action, etc.)."
    }
]

In [29]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate.from_template(
  "User input: {question}\n Cypher query: {query}"
)

few_shot_prompt = FewShotPromptTemplate(
  examples= complex_tests_examples[:5],
  example_prompt= example_prompt,
  prefix= "You are a Neo4j expert. Given an input question, create a syntactically very accurate Cypher query",
  suffix= "User input: {question}\nCypher query: ",
  input_variables=["schema", "question"]
)

In [33]:
print(few_shot_prompt.format(question="How many artists are there?", schema="foo"))

You are a Neo4j expert. Given an input question, create a syntactically very accurate Cypher query

User input: Recommend co-actors for Leonardo DiCaprio that he hasn't worked with, but who his co-actors have collaborated with, ordered by collaboration strength.
 Cypher query: 
        MATCH (diCaprio:Person {name: 'Leonardo DiCaprio'})-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(coActors:Person),
        (coActors)-[:ACTED_IN]->(m2:Movie)<-[:ACTED_IN]-(recommended:Person)
        WHERE NOT (diCaprio)-[:ACTED_IN]->()<-[:ACTED_IN]-(recommended) AND diCaprio <> recommended
        RETURN recommended.name AS recommended, count(recommended) AS strength
        ORDER BY strength DESC
        LIMIT 5
        

User input: Find co-actors who could introduce Christopher Nolan to Leonardo DiCaprio through shared movies (but they haven't collaborated directly).
 Cypher query: 
        MATCH (nolan:Person {name: 'Christopher Nolan'})-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(coActor:Person),
        (coActor

In [34]:
graph_chain_1 = GraphCypherQAChain.from_llm(
  graph= graph,
  llm= llm,
  cypher_prompt= few_shot_prompt,
  verbose= True,
  allow_dangerous_requests=True
)

In [35]:
graph_chain_1.invoke("List all the genres of the movie Interstellar.")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (m:Movie {title: 'Interstellar'})-[:IN_GENRE]->(g:Genre)
RETURN g.name AS genre
ORDER BY g.name

Full Context:
[{'genre': 'Adventure'}, {'genre': 'Drama'}, {'genre': 'Sci-Fi'}]

> Finished chain.


{'query': 'List all the genres of the movie Interstellar.',
 'result': 'Adventure, Drama, Sci‑Fi.'}

In [38]:
graph_chain_1.invoke("List all actors who acted in multiple movies along with the movies they have worked in.")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
// Actors who have acted in more than one distinct movie
MATCH (actor:Person)-[:ACTED_IN]->(movie:Movie)
WITH actor,
     collect(DISTINCT movie.title) AS movies,
     count(DISTINCT movie)        AS movieCount
WHERE movieCount > 1
RETURN actor.name      AS actor,
       movies          AS movies,
       movieCount      AS movieCount
ORDER BY movieCount DESC, actor ASC;

Full Context:
[{'actor': 'Robert Downey Jr.', 'movies': ['The Avengers', 'Iron Man', 'Avengers: Endgame'], 'movieCount': 3}, {'actor': 'Chris Evans', 'movies': ['The Avengers', 'Avengers: Endgame'], 'movieCount': 2}, {'actor': 'Joaquin Phoenix', 'movies': ['Gladiator', 'Joker'], 'movieCount': 2}, {'actor': 'Leonardo DiCaprio', 'movies': ['Inception', 'Titanic'], 'movieCount': 2}]

> Finished chain.


{'query': 'List all actors who acted in multiple movies along with the movies they have worked in.',
 'result': 'Robert Downey Jr.: The Avengers, Iron Man, Avengers: Endgame  \nChris Evans: The Avengers, Avengers: Endgame  \nJoaquin Phoenix: Gladiator, Joker  \nLeonardo DiCaprio: Inception, Titanic'}

In [40]:
graph_chain_1.invoke("List all directors who directed multiple movies along with the movies")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
// Find every person who has directed more than one movie
MATCH (director:Person)-[:DIRECTED]->(movie:Movie)
WITH director,
     collect(DISTINCT movie.title) AS movies,
     count(DISTINCT movie)        AS movieCount
WHERE movieCount > 1
RETURN director.name      AS director,
       movieCount          AS numberOfMovies,
       movies               AS moviesDirected
ORDER BY movieCount DESC, director.name;

Full Context:
[{'director': 'Christopher Nolan', 'numberOfMovies': 3, 'moviesDirected': ['Inception', 'The Dark Knight', 'Interstellar']}, {'director': 'David Fincher', 'numberOfMovies': 2, 'moviesDirected': ['Fight Club', 'The Social Network']}, {'director': 'James Cameron', 'numberOfMovies': 2, 'moviesDirected': ['Avatar', 'Titanic']}]

> Finished chain.


{'query': 'List all directors who directed multiple movies along with the movies',
 'result': 'Christopher Nolan – Inception, The Dark Knight, Interstellar  \nDavid Fincher – Fight Club, The Social Network  \nJames Cameron – Avatar, Titanic'}